# M3 Quad + Tet Workflows

This notebook demonstrates:
- 2D quad adaptive remeshing with snapshots
- 3D tet setup on sphere-like geometry
- diagnostics and JAX optimization loop

In [ ]:
import jax
import jax.numpy as jnp
from pathlib import Path

from gmshjax.mesh.topology import unit_square_quad_mesh, unit_cube_tet_mesh
from gmshjax.mesh.adaptive_quad import adaptive_remesh_quad
from gmshjax.mesh.adaptive_tet import adaptive_remesh_tet
from gmshjax.mesh.generators import project_cube_points_to_sphere
from gmshjax.mesh.diagnostics import quad_diagnostics, tet_diagnostics
from gmshjax.mesh.operators import quad_icn

In [ ]:
# Quad adaptive remesh
qt, qp = unit_square_quad_mesh(16, 12)
qbuf, qhist = adaptive_remesh_quad(
    qp,
    qt.elements,
    max_nodes=6000,
    max_elements=10000,
    max_iters=4,
    target_area=0.01,
    target_mean_icn=0.5,
    snapshot_dir=Path('results') / 'nb_quad_adapt',
)
qstats = quad_diagnostics(qbuf.points[:qbuf.node_count], qbuf.elements[:qbuf.element_count])
qstats

In [ ]:
# Tet setup and adaptive pass on sphere-projected mesh
tt, tp = unit_cube_tet_mesh(6, 6, 5)
sp = project_cube_points_to_sphere(tp, radius=1.0)
tbuf, thist = adaptive_remesh_tet(
    sp,
    tt.elements,
    max_nodes=12000,
    max_elements=24000,
    max_iters=2,
    target_volume=0.05,
    target_mean_icn=0.0,
    snapshot_dir=Path('results') / 'nb_tet_adapt',
)
tstats = tet_diagnostics(tbuf.points[:tbuf.node_count], tbuf.elements[:tbuf.element_count])
tstats

In [ ]:
# JAX optimization on quad quality
topo, points = unit_square_quad_mesh(22, 16)
x = points[:, 0]
y = points[:, 1]
points = points.at[:, 1].set(y + 0.1 * jnp.sin(2 * jnp.pi * x))

def loss_fn(p):
    q = quad_icn(p, topo.elements)
    return jnp.mean((0.8 - q) ** 2)

vag = jax.jit(jax.value_and_grad(loss_fn))
p = points
for _ in range(50):
    val, g = vag(p)
    p = p - 0.03 * g

print('final_loss', float(val))
print('final_mean_icn', float(jnp.mean(quad_icn(p, topo.elements))))